# Reject a forbidden transition

Try to return a paid order to draft.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from aoa.action_machine.domain import BaseStateEntity, Lifecycle, value
from aoa.action_machine.domain.lifecycle import InvalidTransitionError


class OrderStateEntity(BaseStateEntity):
    """Describe one possible order status."""

    DRAFT = value(code="draft", display_name="Draft")
    PAID = value(code="paid", display_name="Payment received")


class OrderLifecycle(Lifecycle[OrderStateEntity]):
    """Allow a draft order to become paid."""

    _template = (
        Lifecycle()
        .state(OrderStateEntity.DRAFT)
        .to(OrderStateEntity.PAID)
        .initial()
        .state(OrderStateEntity.PAID)
        .final()
    )

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
current = OrderLifecycle("paid")
assert not current.can_transition("draft")
try:
    current.transition("draft")
except InvalidTransitionError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("A final state accepted a transition")
assert current.current_state == "paid"

InvalidTransitionError: Transition 'paid' → 'draft' is not allowed on OrderLifecycle. Allowed transitions from 'paid': (no transitions).


## What the result means

The template permits draft to become paid but defines no return transition. `transition()` checks the current state’s outgoing edges and refuses paid-to-draft. Constructing a lifecycle from a known code is a different operation; it does not establish transition history.